# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load and explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`. This step retrieves the schema and enables exploration of record sets, fields, and records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Retrieve metadata (schema)
metadata = dataset.metadata

# Print main dataset information
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Dataset @id: {metadata.id}")
print(f"Date Published: {metadata.date_published}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review the available record sets and their fields by referencing their `@id`. We use the `dataset.record_sets` property to get a list of record sets, and display their `@id` and associated field IDs.

In [ ]:
# List all record sets and their fields
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id} (Type: {field.data_type})")
    print()

# Example: Preview records in each record set
for rs in record_sets:
    print(f"Records from RecordSet '{rs.name}' (@id: {rs.id}):")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        if i < 2:
            print(record)
        else:
            break
    print()

## 3. Data Extraction
Load data from the identified record sets into pandas DataFrames for further analysis. Use the `@id` of each record set as discovered above.

In [ ]:
# Store DataFrames per RecordSet (@id)
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded RecordSet @id: {rs_id}, shape: {df.shape}")

# Display columns from first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in RecordSet @id '{first_rs_id}':\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All operations reference field columns by their `@id` as defined in record sets.

In [ ]:
# Identify numeric fields from first record set
import numpy as np
rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(rs_id, pd.DataFrame())

numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
print(f"Numeric fields in RecordSet @id '{rs_id}': {numeric_fields}")

# If numeric field exists, filter, normalize, and group
if numeric_fields:
    numeric_field = numeric_fields[0]  # Use first numeric field
    threshold = df[numeric_field].mean()  # Use mean as threshold example
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping using a categorical field
    categorical_fields = [col for col in df.columns if df[col].dtype == object]
    group_field = categorical_fields[0] if categorical_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields. All data references to columns use the `@id` from the record set.

In [ ]:
# Visualize numeric field distribution
if numeric_fields:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=15, color='skyblue')
    plt.title(f"Distribution of '{numeric_field}' in RecordSet @id '{rs_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize normalized field
    plt.figure(figsize=(8, 4))
    filtered_df[f"{numeric_field}_normalized"].hist(bins=15, color='salmon')
    plt.title(f"Distribution of Normalized '{numeric_field}' After Filtering")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel('Frequency')
    plt.show()

    # Grouped bar plot
    if group_field:
        grouped_df.plot.bar(x=group_field, y=numeric_field, color='mediumseagreen')
        plt.title(f"Grouped Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric fields for visualization.")

## 6. Conclusion

This notebook demonstrated loading and exploring the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id`. We overviewed available record sets and fields, loaded records into DataFrames, and performed basic EDA & visualizations. For detailed analysis or modeling, refer to field descriptions from the Croissant schema and expand the exploratory steps as needed.